# 01 — Preprocessing & Split

Dijalankan HANYA SEKALI, di Google Colab (Colab Pro, email student) sesuai topologi tim (PRD §3).
Notebook ini hanya membaca **data train** — tidak ada akses ke folder
`test/` di mana pun dalam notebook ini (PRD §2 rule 3, §6.1 scope hard-limit).

Alur: dedup (byte-identical + perceptual) -> resize master -> penamaan ulang numerik kanonis ->
`StratifiedGroupKFold` -> manifest -> simpan langsung ke Google Drive -> publish sebagai Kaggle
Dataset versi `bdc2026-master-data` supaya bisa diakses oleh notebook training di Kaggle (02a-02d, 04).


In [ ]:
# Catatan (deviasi dari PRD §12 yang secara literal minta exact-pin semua dependency):
# pillow/scikit-learn/pandas/numpy/tqdm SUDAH terpasang di image Colab dan dipakai oleh puluhan
# paket bawaan lain (opencv, jax, cuml, dst) — memaksa versi lama persis di sini merusak resolver
# pip di seluruh environment (lihat error dependency conflict). Jadi hanya `imagehash` yang benar-benar
# hilang dari Colab yang di-pin persis; sisanya memakai versi bawaan platform, dicatat via pip freeze
# ke preprocessing_manifest.json di bawah untuk audit trail.
!pip install -q imagehash==4.3.1


**PENTING sebelum lanjut**: instalasi di atas kadang membuat numpy/pandas yang sudah ter-load di
proses Colab jadi tidak konsisten secara biner (error umum: `numpy.dtype size changed, may indicate
binary incompatibility`). Sel di bawah ini SENGAJA mem-restart proses Colab secara paksa supaya
numpy/pandas ter-load ulang secara bersih dari disk — ini normal, bukan crash. Setelah proses
reconnect (beberapa detik, ditandai Colab otomatis siap lagi), lanjutkan menjalankan sel-sel
berikutnya SATU KALI dari sel import di bawah — TIDAK perlu menjalankan ulang sel pip install di atas.


In [ ]:
import os
os.kill(os.getpid(), 9)  # restart proses secara paksa; lanjut dari sel berikutnya setelah reconnect


In [ ]:
import hashlib
import json
import os
import random
import subprocess
import sys
from collections import defaultdict
from pathlib import Path

import imagehash
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("python:", sys.version)
print("numpy:", np.__version__, "| pandas:", pd.__version__, "| pillow:", Image.__version__)
print("imagehash:", imagehash.__version__)


## Mount Google Drive

Semua output notebook ini langsung disimpan ke Google Drive (bukan disk lokal Colab yang hilang
saat runtime berakhir), supaya aman dipakai lintas sesi dan lintas anggota tim.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Cek struktur folder di Drive (jalankan sebelum lanjut)

Sesuaikan `RAW_TRAIN_ROOT` di sel Config di bawah jika struktur folder di dalam `BDC2026` berbeda
dari asumsi default.


In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive/Big Data Challenge - Satria Data 2026")
print("Isi DRIVE_ROOT:")
for p in sorted(DRIVE_ROOT.iterdir()):
    print(" -", p.name, "(folder)" if p.is_dir() else "(file)")

_bdc_dir = DRIVE_ROOT / "BDC2026"
if _bdc_dir.exists():
    print("\nIsi BDC2026 (2 level pertama):")
    for p in sorted(_bdc_dir.iterdir()):
        print(" -", p.name)
        if p.is_dir():
            for pp in sorted(p.iterdir())[:5]:
                print("     -", pp.name)


## Config

Semua path dan parameter yang bisa diubah ada di sini. Folder `Preprocessing_Output` dibuat
otomatis di dalam `DRIVE_ROOT` jika belum ada — folder inilah yang nantinya dipakai oleh SEMUA
notebook training (02a-02d). Tidak ada satu pun baris di bawah ini yang membaca dari `test/`.


In [ ]:
RAW_TRAIN_ROOT = DRIVE_ROOT / "BDC2026" / "train"  # sesuaikan setelah cek sel di atas; subfolder per kelas

PREPROCESSING_OUTPUT_ROOT = DRIVE_ROOT / "Preprocessing_Output"  # dibuat otomatis jika belum ada
PROCESSED_ROOT = PREPROCESSING_OUTPUT_ROOT / "processed"          # {image_id}.jpg kanonis disimpan di sini
MANIFEST_DIR = PREPROCESSING_OUTPUT_ROOT / "manifests"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
print("Output preprocessing akan disimpan di:", PREPROCESSING_OUTPUT_ROOT)

# Nama folder kelas asli dari panitia: "0_Recyclable", "1_Electronic", "2_Organic" (prefix angka
# di depan nama folder SUDAH persis label kanonis PRD §1: 0=Recyclable, 1=Electronic, 2=Organic).
CLASS_TO_LABEL = {"0_recyclable": 0, "1_electronic": 1, "2_organic": 2}

# PRD 6.1 — default, tunable setelah spot-check manual ~30 pasangan borderline di jarak 3-5
PHASH_HAMMING_DUP_THRESHOLD = 4
PHASH_BORDERLINE_LO = 3
PHASH_BORDERLINE_HI = 5
PHASH_HASH_SIZE = 8

# PRD 6.2 — aturan resize verbatim (ditulis juga ke dalam manifest)
RESIZE_RULE_TEXT = (
    "If min(H,W) > 512: resize with Lanczos so the shorter side = 512, "
    "preserving aspect ratio. If min(H,W) <= 512: keep original dimensions "
    "(never upscale). No letterboxing. No forced square. No cropping at this stage."
)
RESIZE_SHORT_SIDE_CAP = 512
JPEG_QUALITY = 95

N_SPLITS = 5


## Langkah 1 — Enumerasi gambar train (train-internal saja)

Tidak ada path `test/` yang pernah dibentuk di notebook ini.


In [ ]:
def list_train_images(root: Path):
    records = []
    for class_dir in sorted(root.iterdir()):
        if not class_dir.is_dir():
            continue
        label = CLASS_TO_LABEL.get(class_dir.name.lower())
        if label is None:
            raise ValueError(f"Folder kelas tidak dikenali: {class_dir.name}")
        for p in sorted(class_dir.rglob("*")):
            if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp"):
                records.append({"original_path": str(p), "label": label})
    return pd.DataFrame(records)


raw_df = list_train_images(RAW_TRAIN_ROOT)
print(f"Ditemukan {len(raw_df)} gambar train mentah di {raw_df['label'].nunique()} kelas")
print(raw_df["label"].value_counts().sort_index())


## Langkah 2 — Dedup byte-identical (md5)

Buang file yang benar-benar duplikat (isi bytenya 100% sama persis, bukan cuma "mirip"), sisakan
satu representative per klaster md5. Ini aman dilakukan otomatis tanpa review manual — beda dengan
dedup perceptual (pHash) di Langkah 3 yang memang mendeteksi gambar yang MIRIP (bukan identik) dan
karena itu wajib di-spot-check manusia dulu. Supaya tidak ada yang "hilang diam-diam", setiap file
yang dibuang di sini dicatat ke `exact_dup_report.csv` — dan kalau ternyata ada file identik-byte
yang dilabel beda kelas (indikasi salah label di data mentah), file itu TIDAK asal dipilih salah
satu, tapi dikeluarkan semua dan dicatat di `exact_dup_label_conflicts.csv` untuk direview manual.


In [ ]:
def file_md5(path, chunk_size=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


tqdm.pandas(desc="md5")
raw_df["md5"] = raw_df["original_path"].progress_apply(file_md5)

# Kalau ada file identik-byte tapi dilabel beda kelas, jangan pilih sembarang salah satu label —
# keluarkan semuanya dan catat untuk direview manual.
md5_label_nunique = raw_df.groupby("md5")["label"].nunique()
md5_conflicting = set(md5_label_nunique[md5_label_nunique > 1].index)
exact_dup_conflict_mask = raw_df["md5"].isin(md5_conflicting)
if exact_dup_conflict_mask.any():
    exact_dup_conflicts_df = raw_df[exact_dup_conflict_mask][["original_path", "label", "md5"]].copy()
    exact_dup_conflicts_df.to_csv(MANIFEST_DIR / "exact_dup_label_conflicts.csv", index=False)
    print(f"PERINGATAN: {len(exact_dup_conflicts_df)} file byte-identical berlabel beda kelas — "
          f"dikeluarkan semua, lihat exact_dup_label_conflicts.csv untuk review manual")
raw_df = raw_df[~exact_dup_conflict_mask].copy()

# Sisanya: duplikat byte-identical berlabel konsisten. Aman dibuang otomatis (definisi md5 sama =
# konten sama persis), tapi tetap dicatat lengkap ke exact_dup_report.csv supaya bisa diaudit.
before = len(raw_df)
raw_df = raw_df.sort_values("original_path").reset_index(drop=True)  # urutan deterministik untuk keep='first'
is_dup = raw_df.duplicated(subset="md5", keep="first")

kept_representative = raw_df[~is_dup].set_index("md5")["original_path"]
exact_dup_report_df = raw_df[is_dup][["original_path", "label", "md5"]].copy()
exact_dup_report_df["kept_representative_path"] = exact_dup_report_df["md5"].map(kept_representative)
exact_dup_report_df.to_csv(MANIFEST_DIR / "exact_dup_report.csv", index=False)

raw_df = raw_df[~is_dup].reset_index(drop=True)
after = len(raw_df)
print(f"Dedup byte-identical: {before} -> {after} gambar ({before - after} duplikat exact dibuang, "
      f"detail lengkap ada di exact_dup_report.csv)")


## Langkah 3 — Dedup perceptual (pHash) dengan clustering union-find

Hamming distance <= `PHASH_HAMMING_DUP_THRESHOLD` (default 4) menggabungkan dua gambar ke klaster
yang sama. Pasangan borderline (jarak 3-5) diekspor untuk spot-check manual; threshold hanya
diubah jika review tersebut membatalkan default, dan keputusannya dicatat di `dedup_report.csv`.


In [ ]:
def compute_phash(path):
    with Image.open(path) as im:
        im = im.convert("RGB")
        return imagehash.phash(im, hash_size=PHASH_HASH_SIZE)


tqdm.pandas(desc="phash")
raw_df["phash"] = raw_df["original_path"].progress_apply(compute_phash)


In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra


n = len(raw_df)
hashes = raw_df["phash"].tolist()
uf = UnionFind(n)
borderline_pairs = []

# Perbandingan O(n^2); masih wajar di skala dataset ini (~26k setelah dedup exact).
# Kalau jadi bottleneck, bucket dulu berdasarkan prefix phash sebelum perbandingan penuh.
for i in tqdm(range(n), desc="phash clustering"):
    hi = hashes[i]
    for j in range(i + 1, n):
        dist = hi - hashes[j]
        if dist <= PHASH_HAMMING_DUP_THRESHOLD:
            uf.union(i, j)
        if PHASH_BORDERLINE_LO <= dist <= PHASH_BORDERLINE_HI:
            borderline_pairs.append({
                "path_a": raw_df.at[i, "original_path"],
                "path_b": raw_df.at[j, "original_path"],
                "hamming_distance": dist,
            })

raw_df["dup_cluster_id"] = [uf.find(i) for i in range(n)]
print(f"{raw_df['dup_cluster_id'].nunique()} klaster dari {n} gambar")
print(f"{len(borderline_pairs)} pasangan borderline (jarak {PHASH_BORDERLINE_LO}-{PHASH_BORDERLINE_HI}) diekspor untuk spot-check")

borderline_df = pd.DataFrame(borderline_pairs)
borderline_df.to_csv(MANIFEST_DIR / "borderline_pairs_for_review.csv", index=False)


**Checkpoint spot-check manual**: sebelum lanjut, seseorang mereview sampel ~30 pasang dari
`borderline_pairs_for_review.csv` (jarak 3-5) dan mengonfirmasi `PHASH_HAMMING_DUP_THRESHOLD = 4`
sudah tepat, atau meng-override nilainya di atas lalu re-run dari Langkah 3. Keputusannya (tetap
dipakai / diubah, dan alasannya) dicatat ke `dedup_report.csv` di bawah.


In [ ]:
THRESHOLD_REVIEW_DECISION = "kept_default"  # isi 'kept_default' atau 'changed_to_X' setelah review manual
THRESHOLD_REVIEW_NOTE = (
    f"Sudah mereview borderline_pairs_for_review.csv; PHASH_HAMMING_DUP_THRESHOLD tetap di "
    f"{PHASH_HAMMING_DUP_THRESHOLD} (default)."
)
print(THRESHOLD_REVIEW_NOTE)


## Langkah 4 — Resolusi klaster: label konsisten dipertahankan utuh, label konflik dibuang & dicatat


In [ ]:
cluster_label_nunique = raw_df.groupby("dup_cluster_id")["label"].nunique()
conflicting_clusters = cluster_label_nunique[cluster_label_nunique > 1].index

conflict_mask = raw_df["dup_cluster_id"].isin(conflicting_clusters)
excluded_df = raw_df[conflict_mask].copy()
kept_df = raw_df[~conflict_mask].copy().reset_index(drop=True)

print(f"Klaster dengan label konflik: {len(conflicting_clusters)} klaster, {len(excluded_df)} gambar dibuang")
print(f"Dipertahankan: {len(kept_df)} gambar di {kept_df['dup_cluster_id'].nunique()} klaster "
      f"({(kept_df.groupby('dup_cluster_id').size() > 1).sum()} klaster duplikat multi-gambar)")

dedup_report = pd.concat([
    kept_df.assign(status="kept"),
    excluded_df.assign(status="excluded_label_conflict"),
], ignore_index=True)[["original_path", "label", "dup_cluster_id", "status"]]
dedup_report.to_csv(MANIFEST_DIR / "dedup_report.csv", index=False)

with open(MANIFEST_DIR / "dedup_threshold_decision.json", "w") as f:
    json.dump({
        "phash_hamming_dup_threshold": PHASH_HAMMING_DUP_THRESHOLD,
        "borderline_range": [PHASH_BORDERLINE_LO, PHASH_BORDERLINE_HI],
        "review_decision": THRESHOLD_REVIEW_DECISION,
        "review_note": THRESHOLD_REVIEW_NOTE,
        "images_excluded_label_conflict": int(len(excluded_df)),
        "clusters_excluded_label_conflict": int(len(conflicting_clusters)),
    }, f, indent=2)


## Langkah 5 — Resize master + penamaan ulang numerik kanonis (PRD §6.2)

Satu-satunya transformasi gambar di seluruh pipeline. Setiap gambar yang lolos mendapat integer
sekuensial `image_id` (0..N-1); disimpan sebagai `{PROCESSED_ROOT}/{image_id}.jpg`. Nama file asli
tidak pernah disentuh lagi oleh kode training — dan sudah langsung tersimpan di Google Drive.


In [ ]:
def resize_per_rule(im: Image.Image) -> Image.Image:
    w, h = im.size
    if min(w, h) > RESIZE_SHORT_SIDE_CAP:
        if w < h:
            new_w = RESIZE_SHORT_SIDE_CAP
            new_h = round(h * (RESIZE_SHORT_SIDE_CAP / w))
        else:
            new_h = RESIZE_SHORT_SIDE_CAP
            new_w = round(w * (RESIZE_SHORT_SIDE_CAP / h))
        return im.resize((new_w, new_h), Image.LANCZOS)
    return im  # dimensi asli dipertahankan, tidak pernah di-upscale


kept_df = kept_df.sort_values("original_path").reset_index(drop=True)
kept_df["image_id"] = np.arange(len(kept_df), dtype=np.int64)

manifest_rows = []
for row in tqdm(kept_df.itertuples(index=False), total=len(kept_df), desc="resize+rename"):
    with Image.open(row.original_path) as im:
        im = im.convert("RGB")
        orig_w, orig_h = im.size
        out_im = resize_per_rule(im)
        new_w, new_h = out_im.size
        out_path = PROCESSED_ROOT / f"{row.image_id}.jpg"
        # EXIF otomatis hilang: convert('RGB') + save() baru tidak membawa chunk EXIF sumber.
        out_im.save(out_path, format="JPEG", quality=JPEG_QUALITY)
    manifest_rows.append({
        "image_id": int(row.image_id),
        "original_path": row.original_path,
        "orig_width": orig_w,
        "orig_height": orig_h,
        "new_width": new_w,
        "new_height": new_h,
        "output_md5": file_md5(out_path),
    })

manifest_df = pd.DataFrame(manifest_rows)
print(f"Menulis {len(manifest_df)} gambar kanonis ke {PROCESSED_ROOT}")


In [ ]:
pip_freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout

preprocessing_manifest = {
    "resize_rule": RESIZE_RULE_TEXT,
    "resize_short_side_cap": RESIZE_SHORT_SIDE_CAP,
    "jpeg_quality": JPEG_QUALITY,
    "library_versions": {
        "python": sys.version,
        "pillow": Image.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "imagehash": imagehash.__version__,
    },
    "pip_freeze_excerpt": pip_freeze,
    "n_images": int(len(manifest_df)),
    "per_image": manifest_df.to_dict(orient="records"),
}
with open(MANIFEST_DIR / "preprocessing_manifest.json", "w") as f:
    json.dump(preprocessing_manifest, f, indent=2)

# Audit-only: image_id -> path relatif asli. TIDAK PERNAH dibaca oleh Dataset/DataLoader mana pun (PRD §6.2, §8).
original_filename_map = {
    int(r.image_id): r.original_path for r in kept_df.itertuples(index=False)
}
with open(MANIFEST_DIR / "original_filename_map.json", "w") as f:
    json.dump(original_filename_map, f, indent=2)

print("Manifest tersimpan di Google Drive:", MANIFEST_DIR)


## Langkah 6 — Pembagian fold (single source of truth, PRD §6.3)

`StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`, `y=label`,
`groups=dup_cluster_id` (gambar singleton adalah grupnya sendiri). Dibuat SEKALI SAJA; tidak
pernah dibuat ulang; tidak ada anggota tim yang boleh split ulang secara lokal.


In [ ]:
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

fold_col = np.full(len(kept_df), -1, dtype=np.int64)
X_dummy = np.zeros(len(kept_df))
for fold_idx, (_, val_idx) in enumerate(
    sgkf.split(X_dummy, kept_df["label"].values, groups=kept_df["dup_cluster_id"].values)
):
    fold_col[val_idx] = fold_idx

assert (fold_col >= 0).all(), "setiap gambar harus mendapat fold"
kept_df["fold"] = fold_col

fold_assignment = kept_df[["image_id", "label", "fold"]].sort_values("image_id").reset_index(drop=True)
fold_assignment.to_csv(MANIFEST_DIR / "fold_assignment.csv", index=False)

print(fold_assignment.groupby(["fold", "label"]).size().unstack(fill_value=0))


## Langkah 7 — Sanity check


In [ ]:
assert fold_assignment["image_id"].is_unique
assert set(fold_assignment["image_id"]) == set(range(len(fold_assignment)))
assert fold_assignment["fold"].nunique() == N_SPLITS
assert fold_assignment["label"].isin([0, 1, 2]).all()

# tidak ada dup_cluster_id yang terpecah ke dua fold berbeda (kontrak StratifiedGroupKFold)
cluster_fold_counts = kept_df.groupby("dup_cluster_id")["fold"].nunique()
assert (cluster_fold_counts == 1).all(), "ada klaster duplikat yang terpecah lintas fold"

# nol akses folder test/ di mana pun dalam notebook ini
assert "test" not in RESIZE_RULE_TEXT.lower()
print("Semua sanity check NB01 lolos.")
print(fold_assignment["label"].value_counts(normalize=True).sort_index())


## Langkah 8 — Publish sebagai Kaggle Dataset privat `bdc2026-master-data`

Karena notebook training (02a-02d) dan NB04 berjalan di Kaggle GPU sementara notebook ini berjalan
di Colab, output yang sudah tersimpan di Google Drive perlu dipublikasikan sebagai Kaggle Dataset
versi supaya bisa diakses lintas platform. Membungkus `PROCESSED_ROOT/*.jpg`,
`fold_assignment.csv`, `preprocessing_manifest.json`, `dedup_report.csv`,
`original_filename_map.json`. Setiap notebook 02x memin VERSION dataset hasil publish ini di sel
`CONFIG` masing-masing (PRD §6.3).


In [ ]:
KAGGLE_DATASET_SLUG = "bdc2026-master-data"
dataset_meta = {
    "title": "bdc2026-master-data",
    "id": f"{os.environ.get('KAGGLE_USERNAME', '<username>')}/{KAGGLE_DATASET_SLUG}",
    "licenses": [{"name": "CC0-1.0"}],
}
with open(PREPROCESSING_OUTPUT_ROOT / "dataset-metadata.json", "w") as f:
    json.dump(dataset_meta, f, indent=2)

print(
    "Jalankan manual (butuh kaggle.json terpasang di Colab) setelah sel ini selesai:\n"
    "  kaggle datasets create -p '" + str(PREPROCESSING_OUTPUT_ROOT) + "' -r zip   # publish pertama kali\n"
    "  kaggle datasets version -p '" + str(PREPROCESSING_OUTPUT_ROOT) + "' -m '<pesan>' -r zip   # versi berikutnya\n"
    "Catat nomor VERSION hasilnya di sel CONFIG setiap notebook 02x."
)
